# **Supplemental: Plotting a spatial map of lake surface temperature from GLSEA**

In this supplemental script, we will download, read, and visualize a spatial map of lake surface temperature from the Great Lakes Surface Environmental Analysis (GLSEA).

First, we will import necessary libraries.

In [ ]:
# import necessary libraries
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime
import os


# set this False if you are using a temporary folder
use_google_drive = True

if use_google_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    save_dir = "/content/drive/MyDrive/GL_env_data/supplemental"
else:
    save_dir = "/content/GL_env_data/supplemental"



Next, we will download a GLSEA data from the CoastWatch Great Lakes node THREDDS server
*https://coastwatch.glerl.noaa.gov/satellite-data-products/great-lakes-surface-environmental-analysis-glsea/*

In [ ]:
# Create folder if it does not exist
os.makedirs(save_dir, exist_ok=True)

# set URL for GLSEA THREDDS server
url_glsea="https://apps.glerl.noaa.gov/thredds/fileServer/glsea_nc_3"

# set day, month, and year
date = datetime(2026,5,10)
yearstr=str(date.year).zfill(4)
monstr=str(date.month).zfill(2)
doystr=date.strftime("%j").zfill(3)


# set file name
fname=yearstr+"_"+doystr+"_glsea_sst.nc"
url_file=url_glsea+"/"+yearstr+"/"+monstr+"/"+fname
#print(url_file)

# download using the wget command, and move it to the specified directory
os.system('wget '+url_file)
os.system('mv '+fname+' '+save_dir+'/')




In [ ]:
# open the netcdf file using xarray
ds=xr.open_dataset(os.path.join(save_dir, fname))

# print data info
ds

Let's make a quick plot.

In [ ]:
# plot using xarray's plot function
ds['sst'].plot()

We can create a nicer map using Geopandas and basemap.

In [ ]:
# import additional libraries
!pip -q install contextily
import geopandas as gpd
from shapely.geometry import box
import contextily as cx
import matplotlib.pyplot as plt
from shapely.geometry import box
from pyproj import Transformer
from matplotlib.colors import Normalize


# Select first time step
sst = ds['sst'].isel(time=0).squeeze()
lat = sst["lat"].values
lon = sst["lon"].values

# Clean / mask SST data
sst_data = sst.values.astype(float)

lat_min = lat.min()
lat_max = lat.max()
lon_min = lon.min()
lon_max = lon.max()


# GeoPandas bounding box in WGS84
extent_gdf = gpd.GeoDataFrame(
    geometry=[box(lon_min, lat_min, lon_max, lat_max)],
    crs="EPSG:4326")

# Convert extent to Web Mercator for Contextily
extent_3857 = extent_gdf.to_crs(epsg=3857)
xmin, ymin, xmax, ymax = extent_3857.total_bounds

# Reproject lon/lat grid to Web Mercator
lon2d, lat2d = np.meshgrid(lon, lat)

transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:3857",
    always_xy=True)

x2d, y2d = transformer.transform(lon2d, lat2d)

# Plot
fig, ax = plt.subplots(figsize=(11, 9))
# Set map extent before adding basemap
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# Add Contextily basemap
cx.add_basemap(
    ax,
    source=cx.providers.CartoDB.Positron,
    attribution_size=7)

# Plot SST raster
mesh = ax.pcolormesh(
    x2d,y2d,
    sst_data, cmap="turbo")#,
    #shading="auto",
    #alpha=0.88)


print(sst_data)



# Colorbar
cbar = fig.colorbar(mesh, ax=ax, shrink=0.75, pad=0.02)
cbar.set_label(f"Lake surface temperature [degC]")

# Title
time_val = str(ds['time'].isel(time=0).values)
ax.set_title(
    f"GLSEA Lake Surface Temperature\n{time_val}",
    fontsize=14
)

# Clean map axes
ax.set_axis_off()

plt.tight_layout()

We can zoom over the western Lake Erie area, and overlay the regular, structured mesh.

In [ ]:
# Western Lake Erie bounding box, lon/lat
wle_lon_min, wle_lat_min, wle_lon_max, wle_lat_max = (-83.6, 41.3, -82.2, 42.2)

# GeoPandas bounding box in WGS84
extent_gdf_wle = gpd.GeoDataFrame(
    geometry=[box(wle_lon_min, wle_lat_min, wle_lon_max, wle_lat_max)],
    crs="EPSG:4326")

# Convert extent to Web Mercator for Contextily
extent_3857_wle = extent_gdf_wle.to_crs(epsg=3857)
xmin, ymin, xmax, ymax = extent_3857_wle.total_bounds


# Plot
fig, ax = plt.subplots(figsize=(11, 9))
# Set map extent before adding basemap
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# Add Contextily basemap
cx.add_basemap(
    ax,
    source=cx.providers.CartoDB.Positron,
#    source=cx.providers.Esri.WorldImagery,
    attribution_size=7)

# Plot SST raster
mesh = ax.pcolormesh(
    x2d,y2d,
    sst_data, cmap="turbo",
    edgecolors=(1, 1, 1, 0.35),
    linewidth=0.08,
    alpha=0.8)

# Colorbar
cbar = fig.colorbar(mesh, ax=ax, shrink=0.75, pad=0.02)
cbar.set_label(f"Lake surface temperature [degC]")

# Title
time_val = str(ds['time'].isel(time=0).values)
ax.set_title(
    f"GLSEA Lake Surface Temperature\n{time_val}",
    fontsize=14
)

# Clean map axes
ax.set_axis_off()

plt.tight_layout()

##**Further readings and data sources**

* FVCOM Github repository: https://github.com/FVCOM-GitHub/FVCOM

* FVCOM User manual: https://etchellsfleet27.com/wp-content/uploads/2020/06/FVCOM_User_Manual_v3.1.6.pdf


*   SCHISM utility scripts by James Kessler at NOAA Great Lakes Environmental Research Lab: https://github.com/NOAA-GLERL/SCHISM_grid_utils. Can be adaptable for other unstructured mesh model outputs, such as FVCOM.


* NOAA National Ocean Service Operational Forecast Systems: https://tidesandcurrents.noaa.gov/models.html

* NOAA National Data Buoy Center:
  https://www.ndbc.noaa.gov/

* NOAA CoastWatch Great Lakes Regional Node: https://coastwatch.glerl.noaa.gov/

* NOAA GLERL Great Lakes Surface Environmental Analysis:
  https://coastwatch.glerl.noaa.gov/glsea/

* NOAA NOS OFS public data archive on AWS:
https://registry.opendata.aws/noaa-ofs/